In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

from torch.utils.data import Dataset,DataLoader

from tokenizers import ByteLevelBPETokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

from torchinfo import summary
import math


Рассказчик (главный агент, который контролирует правила симуляции)

In [49]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=20000,
    special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"],
    min_frequency=2
)

files = ["VocabText.txt"]
tokenizer.train(files, trainer)
tokenizer.save("tokenizer.json")


In [ ]:
class StoryTeller(nn.Module):
    def __init__(self, window_size: int, vocab_size: int, emb_size: int, rules_count: int):
        super(StoryTeller, self).__init__()

        self.time_window = nn.Sequential(
            nn.Linear(window_size, emb_size),
            nn.Linear(emb_size, rules_count)
        )

        self.chat_inflation = nn.Sequential(
                    nn.Linear(emb_size, rules_count)
                )
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.summary_layer = nn.Linear(rules_count * 2, rules_count)

    def forward(self, text, history):
        emb = self.embedding(text)
        history = self.time_window(history)
        emb_mean = torch.mean(emb, dim=1)
        chat_inf = self.chat_inflation(emb_mean)


        
        all_feautures = torch.cat((history, chat_inf), dim=1)
        res = self.summary_layer(all_feautures)

        return res


In [ ]:

story_model = StoryTeller(window_size=32, vocab_size=4096, emb_size=256, rules_count=12)

batch_size = 2
seq_len = 128

dummy_text = torch.randint(0, 4096, (batch_size, seq_len), dtype=torch.long)
dummy_history = torch.randn(batch_size, 32, dtype=torch.float)


model_stats = summary(
    story_model, 
    input_data=(dummy_text, dummy_history),
    col_names=["input_size", "output_size", "num_params", "kernel_size"],
    depth=3
)

print(model_stats)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape
StoryTeller                              [2, 128]                  [2, 12]                   --                        --
├─Embedding: 1-1                         [2, 128]                  [2, 128, 256]             1,048,576                 --
├─Sequential: 1-2                        [2, 32]                   [2, 12]                   --                        --
│    └─Linear: 2-1                       [2, 32]                   [2, 256]                  8,448                     --
│    └─Linear: 2-2                       [2, 256]                  [2, 12]                   3,084                     --
├─Sequential: 1-3                        [2, 256]                  [2, 12]                   --                        --
│    └─Linear: 2-3                       [2, 256]                  [2, 12]                   3,084                     --
├─Linear: 1-4 